### Model Experiments

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
from collections import Counter
import pickle

In [0]:
df = spark.table("scientific_trend_analysis.core.arxiv_cleaned_sampled")
display(df)

In [0]:
processed_text_list = [row['processed_text'] for row in df.select('processed_text').collect()]
print(f"Loaded {len(processed_text_list)} processed texts")

In [0]:
tfidf_vectorizer = TfidfVectorizer(max_df=0.85, min_df=5, max_features=30000, sublinear_tf=True, stop_words='english', ngram_range=(1, 2))
X_tfidf = tfidf_vectorizer.fit_transform(processed_text_list)
feature_name = tfidf_vectorizer.get_feature_names_out()
print(feature_name)
print(X_tfidf.shape)

In [0]:
selector = VarianceThreshold(threshold=0.00001)
X_tfidf_reduced = selector.fit_transform(X_tfidf)
print(X_tfidf.shape)
print(X_tfidf_reduced.shape)

In [0]:
n_components = 2000
svd = TruncatedSVD(n_components=n_components, random_state=2)

lsa_result = svd.fit_transform(X_tfidf_reduced)
lsa_result = normalize(lsa_result, norm='l2', axis=1)
print(f"LSA result shape: {lsa_result.shape}")

explained_variance = svd.explained_variance_ratio_.sum()
print(f"Total explained variance: {explained_variance * 100:.2f}%")

In [0]:
# Save LSA features for visualization (numpy array)
dbutils.fs.mkdirs("/Volumes/scientific_trend_analysis/core/model_artifacts/")
with open("/Volumes/scientific_trend_analysis/core/model_artifacts/lsa_features.pkl", "wb") as f:
    pickle.dump(lsa_result, f)

In [0]:
inertia = []
silhouette_scores = {}

k_range = range(6, 16)
for i in k_range:
    km = KMeans(n_clusters=i, 
                init='k-means++',
                n_init='auto',
                random_state=2,
                max_iter=100)
    km.fit(lsa_result)
    inertia.append(km.inertia_)
    cluster_labels = km.predict(lsa_result)
    
    score = silhouette_score(lsa_result, cluster_labels)
    silhouette_scores[i] = score
    print(f"k={i}, Silhouette Score: {score:.4f}")

plt.plot(k_range, inertia, marker='o')
plt.xlabel('Number of cluster', fontsize=14)
plt.ylabel('Inertia', fontsize=14)
plt.xticks(list(k_range))

# find the best k
best_k = max(silhouette_scores, key=silhouette_scores.get)
print(f"\nBest k values according to Silhouette Score: {best_k}")


In [0]:
# final km model
final_kmeans = KMeans(n_clusters=best_k, random_state=2, n_init='auto')
cluster_labels_km = final_kmeans.fit_predict(lsa_result)

# Convert Spark DataFrame to pandas for column assignment
df_label = df.toPandas()
df_label['Cluster_ID_km'] = cluster_labels_km

#Calculate the silhouette score for each data
each_silhouette_score = silhouette_samples(lsa_result,cluster_labels_km,metric="euclidean")
silhouette_avg = silhouette_score(lsa_result,cluster_labels_km)

fig =plt.figure()
ax = fig.add_subplot(1,1,1)
y_lower =10
for i in range(best_k):
    ith_cluster_silhouette_values = each_silhouette_score[cluster_labels_km == i]
    ith_cluster_silhouette_values.sort()
    size_cluster_i = ith_cluster_silhouette_values.shape[0]
    y_upper = y_lower + size_cluster_i

    colors = plt.cm.tab10(range(best_k))
    ax.fill_betweenx(
        np.arange(y_lower, y_upper),
        0,
        ith_cluster_silhouette_values,
        facecolor=colors[i],
        edgecolor=colors[i],
        alpha=0.7
    )
    
    # Label cluster in the middle
    ax.text(-0.05, y_lower + 0.5 * size_cluster_i, str(i), fontsize=10, fontweight='bold')
    
    # Update y_lower for next cluster
    y_lower = y_upper + 10

ax.set_title(f"Silhouette Plot (KMeans k={best_k}, sampled n={len(lsa_result):,})", 
             fontsize=16, fontweight='bold')
ax.set_xlabel("Silhouette Coefficient", fontsize=14)
ax.set_ylabel("Cluster Label", fontsize=14)

# Average silhouette line
ax.axvline(x=silhouette_avg, color="red", linestyle="--", linewidth=2,
           label=f'Average Score: {silhouette_avg:.4f}')

ax.set_yticks([])
ax.set_xticks([-0.2, 0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax.legend(loc='upper right', fontsize=12)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [0]:
text_column = 'processed_text'
df_cluster_text = df_label.groupby('Cluster_ID_km')[text_column].agg(lambda x: ' '.join(x))
df_cluster_text = df_cluster_text.to_frame(name='Combined_Text')

In [0]:
top_n_words = 10 
topic_summary = {}

print("\n--- Top Words for Topic Interpretation ---")

for cluster_id, row in df_cluster_text.iterrows():
    combined_text = row['Combined_Text']
    words = combined_text.split()
    word_counts = Counter(words)
    top_words = word_counts.most_common(top_n_words)
    
    top_words_list = [item[0] for item in top_words]
    topic_summary[cluster_id] = top_words_list
    
    # Combined output on single line
    keywords = ', '.join(top_words_list)
    print(f"Cluster {cluster_id} (Documents: {len(words)} words): {keywords}")

In [0]:
display(df_label)

In [0]:
spark.createDataFrame(df_label).write.mode("overwrite") \
    .saveAsTable("scientific_trend_analysis.rep.arxiv_clustered")
print(f"Saved {len(df_label)} records with cluster assignments")